# Simulation Playground
Use the tabbed sliders to configure every parameter, then click **▶ Run Simulation**.
The sensitivity scan tab runs a λ/γ grid and renders a heatmap.

**One-time setup:** run Cell 1 (imports) and Cell 2 (load data) once per session.
After that, just hit the buttons — no need to re-run earlier cells.

In [ ]:
# ── Cell 1 · Imports ──────────────────────────────────────────────────────────
import math, os, random, sys, time, warnings
warnings.filterwarnings('ignore')
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
import ipywidgets as widgets
from IPython.display import display, clear_output

_HERE = os.path.abspath('')
_REPO = os.path.normpath(os.path.join(_HERE, '..'))
sys.path.insert(0, os.path.join(_REPO, 'Warehouse'))
sys.path.insert(0, _HERE)

from Aisle_Storage import Aisle
from Affinity_Store import AffinityStore
from Inventory_Management import (
    Inventory_Manager, LoadParams,
    build_load_minimizing_assignment_fn,
    build_load_maximizing_assignment_fn,
)
from generate_inventory import load_inventory_from_db
from Pick import PickConfig, PickSimulation
from Warehouse_Builder import AisleConfig, Warehouse_Builder, WarehouseConfig
from Workload_Builder import Batch, BatchConfig, Task
from Simulation_Analytics import extract_batch_stats
from Workload import WorkloadParams
print('Imports OK')

In [ ]:
# ── Cell 2 · Load inventory + affinity (run once per session) ─────────────────
_BATCHES_DIR = os.path.join(_REPO, 'Warehouse', 'generated', 'batches')

def _find_latest_pair(d):
    if not os.path.isdir(d): return None, None
    for b in reversed(sorted(os.listdir(d))):
        bp = os.path.join(d, b)
        if not os.path.isdir(bp): continue
        for p in sorted(os.listdir(bp)):
            pp = os.path.join(bp, p)
            i = os.path.join(pp, 'inventory', 'inventory.db')
            a = os.path.join(pp, 'affinity',  'affinity.db')
            if os.path.exists(i) and os.path.exists(a):
                return i, a
    return None, None

# Override with explicit paths if needed:
_inv_db, _aff_db = _find_latest_pair(_BATCHES_DIR)
# _inv_db = r'C:\path\to\inventory.db'
# _aff_db = r'C:\path\to\affinity.db'

print(f'Inventory : {_inv_db}')
print(f'Affinity  : {_aff_db}')

t0        = time.perf_counter()
inventory = load_inventory_from_db(_inv_db)
n_skus    = len(inventory.cartons)
print(f'{n_skus:,} SKUs  ({time.perf_counter()-t0:.2f}s)')

t0             = time.perf_counter()
affinity_store = AffinityStore(_aff_db)
n_aff = affinity_store._matrix.nnz if affinity_store._matrix is not None else 0
mb    = 0 if affinity_store._matrix is None else (
    affinity_store._matrix.data.nbytes +
    affinity_store._matrix.indices.nbytes +
    affinity_store._matrix.indptr.nbytes) / 1_048_576
print(f'Affinity  : {n_aff:,} entries  {mb:.0f} MB  ({time.perf_counter()-t0:.1f}s)')

# Fixed warehouse layout constants
_BINS_PER_AISLE    = 25 * 20
_N_PALLET_TYPES    = 48
_N_SINGLETON_TYPES = 12
_SINGLETON_MAX_DIM = 16
_CATEGORIES  = ['food', 'clothing', 'electronic', 'furniture', 'seasonal', 'chemical']
_ALL_SIZES   = ['small', 'medium', 'large', 'extra_large']
_CONV_SIZES  = ['small', 'medium', 'large'];        _CONV_PROBS  = [0.25, 0.50, 0.25]
_NCONV_SIZES = ['medium', 'large', 'extra_large'];  _NCONV_PROBS = [0.20, 0.50, 0.30]
_AISLE_CFGS  = []
for _sz in _ALL_SIZES:
    for _cat in _CATEGORIES:
        _AISLE_CFGS.append(AisleConfig('conveyable',     _cat, 'pallet', 25, 20, [_sz], None))
        _AISLE_CFGS.append(AisleConfig('non-conveyable', _cat, 'pallet', 25, 20, [_sz], None))
for _cat in _CATEGORIES:
    _AISLE_CFGS.append(AisleConfig('conveyable',     _cat, 'singleton', 25, 20, _CONV_SIZES,  _CONV_PROBS))
    _AISLE_CFGS.append(AisleConfig('non-conveyable', _cat, 'singleton', 25, 20, _NCONV_SIZES, _NCONV_PROBS))
print('Warehouse layout constants ready.')

In [ ]:
# ── Cell 3 · Widget definitions ───────────────────────────────────────────────
_S = dict(style={'description_width': '155px'}, layout=widgets.Layout(width='400px'))

# Simulation
w_n_batches  = widgets.IntSlider(  value=200,   min=50,    max=2000,  step=50,     description='N batches',      **_S)
w_k_pickers  = widgets.IntSlider(  value=25,    min=5,     max=60,    step=5,      description='K pickers',      **_S)
w_seed_world = widgets.IntSlider(  value=42,    min=0,     max=9999,  step=1,      description='Seed (world)',   **_S)
w_seed_batch = widgets.IntSlider(  value=1337,  min=0,     max=9999,  step=1,      description='Seed (batch)',   **_S)
w_plot_win   = widgets.IntSlider(  value=30,    min=5,     max=200,   step=5,      description='Rolling window', **_S)

# Pick model
w_x_move    = widgets.FloatSlider(value=1.0,   min=0.1,   max=5.0,   step=0.1,    description='x move time',   **_S)
w_y_move    = widgets.FloatSlider(value=0.5,   min=0.1,   max=5.0,   step=0.1,    description='y move time',   **_S)
w_intercept = widgets.FloatSlider(value=1.0,   min=0.1,   max=5.0,   step=0.1,    description='pick intercept',**_S)
w_wt_coef   = widgets.FloatSlider(value=1.1,   min=0.1,   max=5.0,   step=0.1,    description='weight coef',   **_S)
w_vol_coef  = widgets.FloatSlider(value=0.001, min=0.0001,max=0.01,  step=0.0001, description='volume coef',
                                   readout_format='.4f', **_S)
w_cart_coef = widgets.FloatSlider(value=10.0,  min=1.0,   max=50.0,  step=1.0,    description='cart swap coef',**_S)

# Load model  L_a = W_a + λ·(W_a/k)^γ · lift
w_lambda = widgets.FloatSlider(value=1.1, min=0.1, max=5.0, step=0.1, description='λ (lambda)',     **_S)
w_k_load = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='k (pickers/task)',**_S)
w_gamma  = widgets.FloatSlider(value=1.5, min=0.5, max=4.0, step=0.1, description='γ (gamma)',      **_S)

# Batch
w_mean_frac = widgets.FloatSlider(value=0.25, min=0.05, max=0.60, step=0.05, description='mean frac',  **_S)
w_std_frac  = widgets.FloatSlider(value=0.03, min=0.01, max=0.15, step=0.01, description='std frac',   **_S)

# Warehouse sizing
w_sing_cap    = widgets.FloatSlider(value=0.35, min=0.10, max=0.60, step=0.05, description='singleton cap',  **_S)
w_target_fill = widgets.FloatSlider(value=0.90, min=0.50, max=0.99, step=0.01, description='target fill',   **_S)

# Sensitivity scan grid
w_lam_min = widgets.FloatSlider(value=0.5, min=0.1, max=3.0, step=0.1, description='λ min',        **_S)
w_lam_max = widgets.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1, description='λ max',        **_S)
w_lam_n   = widgets.IntSlider(  value=4,   min=2,   max=8,   step=1,   description='λ steps',      **_S)
w_gam_min = widgets.FloatSlider(value=1.0, min=0.5, max=2.5, step=0.1, description='γ min',        **_S)
w_gam_max = widgets.FloatSlider(value=2.5, min=1.0, max=4.0, step=0.1, description='γ max',        **_S)
w_gam_n   = widgets.IntSlider(  value=4,   min=2,   max=8,   step=1,   description='γ steps',      **_S)
w_sens_n  = widgets.IntSlider(  value=30,  min=10,  max=200, step=10,  description='batches/cell', **_S)

print('Widgets defined.')

In [ ]:
# ── Cell 4 · Assemble UI ──────────────────────────────────────────────────────
tab_sim  = widgets.VBox([w_n_batches, w_k_pickers, w_seed_world, w_seed_batch, w_plot_win])
tab_pick = widgets.VBox([w_x_move, w_y_move, w_intercept, w_wt_coef, w_vol_coef, w_cart_coef])
tab_load = widgets.VBox([
    widgets.HTML('<b>L_a = W_a + λ·(W_a/k)^γ · lift_sum</b>'),
    w_lambda, w_k_load, w_gamma,
])
tab_batch = widgets.VBox([w_mean_frac, w_std_frac])
tab_wh    = widgets.VBox([w_sing_cap, w_target_fill])
tab_sens  = widgets.VBox([
    widgets.HTML('<b>Scans λ/γ grid, runs B and C vs A</b>'),
    w_lam_min, w_lam_max, w_lam_n,
    widgets.HTML('<hr style="margin:4px 0">'),
    w_gam_min, w_gam_max, w_gam_n,
    widgets.HTML('<hr style="margin:4px 0">'),
    w_sens_n,
])

tabs = widgets.Tab(children=[tab_sim, tab_pick, tab_load, tab_batch, tab_wh, tab_sens])
for i, name in enumerate(['Simulation', 'Pick Model', 'Load Model', 'Batch', 'Warehouse', 'Sensitivity']):
    tabs.set_title(i, name)

btn_run  = widgets.Button(description='▶  Run Simulation',
                           button_style='success',
                           layout=widgets.Layout(width='200px', height='38px'))
btn_sens = widgets.Button(description='⚡  Sensitivity Scan',
                           button_style='info',
                           layout=widgets.Layout(width='200px', height='38px'))

status      = widgets.HTML(value='<i style="color:grey">Ready — adjust sliders then click Run.</i>')
out_summary = widgets.Output()   # summary table — pinned directly below buttons
out_sim     = widgets.Output()   # plots
out_sens    = widgets.Output()   # sensitivity heatmap

display(widgets.VBox([
    tabs,
    widgets.HBox([btn_run, btn_sens], layout=widgets.Layout(margin='8px 0')),
    status,
    out_summary,
]))

In [ ]:
# ── Cell 5 · Helper functions ─────────────────────────────────────────────────
_COL = {'A \u00b7 Uniform': '#5b9bd5', 'B \u00b7 Load-Min': '#f4a030', 'C \u00b7 Load-Max': '#70ad47'}

def _size_warehouse(sing_cap, target_fill):
    n_singleton   = sum(1 for c in inventory.cartons
                        if max(c.length, c.width, c.height) <= _SINGLETON_MAX_DIM)
    n_pallet      = n_skus - n_singleton
    sing_replicas = max(1, math.ceil(n_singleton / (_N_SINGLETON_TYPES * _BINS_PER_AISLE)))
    pall_replicas = max(1, math.ceil(n_pallet    / (_N_PALLET_TYPES    * _BINS_PER_AISLE)))
    sing_bins_now = _N_SINGLETON_TYPES * sing_replicas * _BINS_PER_AISLE
    min_pall_bins = math.ceil(sing_bins_now * (1 - sing_cap) / sing_cap)
    pall_replicas = max(pall_replicas, math.ceil(min_pall_bins / (_N_PALLET_TYPES * _BINS_PER_AISLE)))
    sing_bins = _N_SINGLETON_TYPES * sing_replicas * _BINS_PER_AISLE
    pall_bins = _N_PALLET_TYPES    * pall_replicas * _BINS_PER_AISLE
    min_total = math.ceil(n_skus / target_fill)
    if sing_bins + pall_bins < min_total:
        extra = math.ceil((min_total - sing_bins - pall_bins) / (_N_PALLET_TYPES * _BINS_PER_AISLE))
        pall_replicas += extra
    total_aisles = _N_SINGLETON_TYPES * sing_replicas + _N_PALLET_TYPES * pall_replicas
    aisle_splits = ([pall_replicas / total_aisles] * _N_PALLET_TYPES +
                    [sing_replicas / total_aisles] * _N_SINGLETON_TYPES)
    return WarehouseConfig(total_aisles=total_aisles,
                           aisle_splits=aisle_splits,
                           aisle_configs=_AISLE_CFGS)

def _build_manager(wh_cfg, seed_layout, seed_stock, affinity=None):
    Aisle.next_aisle_id = 1
    random.seed(seed_layout)
    wh  = Warehouse_Builder().from_config(wh_cfg).build()
    random.seed(seed_stock)
    mgr = Inventory_Manager(wh, affinity=affinity)
    return wh, mgr

def _overstock(mgr, target):
    total  = len(mgr.warehouse.bins)
    needed = round(target * total) - len(mgr.unavailable)
    if needed <= 0: return
    weights = [c.demand.frequency for c in inventory.cartons]
    total_w = sum(weights)
    sample  = random.choices(inventory.cartons,
                              weights=[w / total_w for w in weights], k=needed * 3)
    mgr.enqueue_all(sample, quantity=1)

def _bdf(recs, label):
    return pd.DataFrame([{
        'batch'        : r.batch_id,
        'strategy'     : label,
        'duration'     : r.duration,
        'num_tasks'    : r.num_tasks,
        'total_items'  : r.total_items,
        'throughput'   : r.total_items / r.duration if r.duration > 0 else 0,
        'picking_pct'  : r.picking_pct   * 100,
        'traveling_pct': r.traveling_pct * 100,
    } for r in recs])

def _kde(ax, data, color, label):
    data = np.asarray(data, dtype=float)
    ax.hist(data, bins=30, color=color, alpha=0.6, edgecolor='white')
    if len(data) > 1 and data.max() > data.min():
        kde = gaussian_kde(data, bw_method='silverman')
        xs  = np.linspace(data.min(), data.max(), 300)
        ax.plot(xs, kde(xs) * len(data) * (data.max() - data.min()) / 30,
                color=color, lw=2)
    ax.axvline(data.mean(),     color='red',    lw=1.4, ls='--',
               label=f'Mean   {data.mean():.1f}')
    ax.axvline(np.median(data), color='orange', lw=1.4, ls=':',
               label=f'Median {np.median(data):.1f}')
    ax.set_title(label, fontsize=11); ax.legend(fontsize=9)

print('Helpers defined.')

In [ ]:
# ── Cell 6 · Run simulation callback ──────────────────────────────────────────
def run_simulation(_=None):
    SW  = w_seed_world.value
    SB  = w_seed_batch.value
    NB  = w_n_batches.value
    KP  = w_k_pickers.value
    TF  = w_target_fill.value
    SC  = w_sing_cap.value
    WIN = w_plot_win.value

    pick_cfg = PickConfig(
        num_pickers      = KP,
        x_move_time      = w_x_move.value,
        y_move_time      = w_y_move.value,
        pick_intercept   = w_intercept.value,
        pick_weight_coef = w_wt_coef.value,
        pick_volume_coef = w_vol_coef.value,
        cart_swap_coef   = w_cart_coef.value,
    )
    wp          = WorkloadParams.from_pick_config(pick_cfg)
    load_params = LoadParams(lambda_=w_lambda.value, k=w_k_load.value, gamma=w_gamma.value)
    batch_cfg   = BatchConfig(n_skus, w_mean_frac.value, w_std_frac.value)
    wh_cfg      = _size_warehouse(SC, TF)
    total_bins  = wh_cfg.total_aisles * _BINS_PER_AISLE

    status.value = f'<b>Building warehouses…</b> ({wh_cfg.total_aisles} aisles / {total_bins:,} bins)'

    _, mgr_A = _build_manager(wh_cfg, SW, SW + 100)
    mgr_A.enqueue_all(inventory.cartons, quantity=1)
    _overstock(mgr_A, TF)

    _, mgr_B = _build_manager(wh_cfg, SW, SW + 100, affinity_store)
    mgr_B.enqueue_all(inventory.cartons, quantity=1)
    _overstock(mgr_B, TF)
    mgr_B.init_lift_state(affinity_store)
    mgr_B.assignment_fn = build_load_minimizing_assignment_fn(
        load_params, affinity_store, wp,
        mgr_B._aisle_sku_sets, mgr_B._aisle_lift_sum, mgr_B._aisle_idx_sets)

    _, mgr_C = _build_manager(wh_cfg, SW, SW + 100, affinity_store)
    mgr_C.enqueue_all(inventory.cartons, quantity=1)
    _overstock(mgr_C, TF)
    mgr_C.init_lift_state(affinity_store)
    mgr_C.assignment_fn = build_load_maximizing_assignment_fn(
        load_params, affinity_store, wp,
        mgr_C._aisle_sku_sets, mgr_C._aisle_lift_sum, mgr_C._aisle_idx_sets)

    status.value = f'<b>Simulating {NB} batches…</b>'
    random.seed(SB)
    rA, rB, rC = [], [], []
    skipped = 0
    t0 = time.perf_counter()

    for i in range(NB):
        mgr_A.check_reorders()
        mgr_B.check_reorders()
        mgr_C.check_reorders()
        batch = Batch(batch_cfg, inventory, affinity=affinity_store)
        ta = Task.from_batch(batch, mgr_A.warehouse, manager=mgr_A)
        tb = Task.from_batch(batch, mgr_B.warehouse, manager=mgr_B)
        tc = Task.from_batch(batch, mgr_C.warehouse, manager=mgr_C)
        if not ta or not tb or not tc:
            skipped += 1; continue
        ea = PickSimulation(ta, pick_cfg, manager=mgr_A).run()
        eb = PickSimulation(tb, pick_cfg, manager=mgr_B).run()
        ec = PickSimulation(tc, pick_cfg, manager=mgr_C).run()
        rA.append(extract_batch_stats(ea, i, KP))
        rB.append(extract_batch_stats(eb, i, KP))
        rC.append(extract_batch_stats(ec, i, KP))

    elapsed = time.perf_counter() - t0
    df_A = _bdf(rA, 'A · Uniform')
    df_B = _bdf(rB, 'B · Load-Min')
    df_C = _bdf(rC, 'C · Load-Max')

    # ── Summary table — written to out_summary, pinned below buttons ──────────
    metrics = ['duration', 'throughput', 'picking_pct', 'traveling_pct']
    base    = df_A[metrics].mean()
    rows    = []
    for dfi, lbl in [(df_A, 'A  Uniform'), (df_B, 'B  Load-Min'), (df_C, 'C  Load-Max')]:
        m   = dfi[metrics].mean()
        row = {'strategy': lbl}
        for col in metrics:
            row[col] = round(m[col], 2)
            if lbl != 'A  Uniform':
                d = (m[col] - base[col]) / base[col] * 100 if base[col] != 0 else float('nan')
                row[col + ' Δ%'] = round(d, 2)
        rows.append(row)

    summary_df = pd.DataFrame(rows).set_index('strategy')

    with out_summary:
        clear_output(wait=True)
        print(f'  {len(rA)} batches  |  {elapsed:.1f}s  ({len(rA)/elapsed:.2f}/s)  |  '
              f'{skipped} skipped\n')
        print(summary_df.to_string())

    # ── Plots — written to out_sim ────────────────────────────────────────────
    with out_sim:
        clear_output(wait=True)

        # Plot 1: Rolling mean duration
        fig, ax = plt.subplots(figsize=(12, 3.5))
        for dfi in [df_A, df_B, df_C]:
            lbl  = dfi['strategy'].iloc[0]
            roll = dfi['duration'].rolling(WIN, min_periods=1).mean()
            ax.plot(dfi['batch'], roll, label=lbl, color=_COL[lbl], lw=1.8)
        ax.set_xlabel('Batch'); ax.set_ylabel('Duration')
        ax.set_title(f'Batch duration — {WIN}-batch rolling mean', fontsize=12)
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout(); plt.show()

        # Plot 2: Duration KDE distributions
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        fig.suptitle('Duration Distributions', fontsize=12, fontweight='bold')
        for ax, dfi in zip(axes, [df_A, df_B, df_C]):
            _kde(ax, dfi['duration'].values, _COL[dfi['strategy'].iloc[0]], dfi['strategy'].iloc[0])
            ax.set_xlabel('Duration')
        plt.tight_layout(); plt.show()

        # Plot 3: Picker time split
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        fig.suptitle('Picker Time Split (mean across batches)', fontsize=12)
        x      = np.arange(3)
        lbls   = [df['strategy'].iloc[0] for df in [df_A, df_B, df_C]]
        colors = [_COL[l] for l in lbls]
        for ax, col, title in [
            (axes[0], 'picking_pct',   'Picking %'),
            (axes[1], 'traveling_pct', 'Traveling %'),
        ]:
            vals = [df[col].mean() for df in [df_A, df_B, df_C]]
            bars = ax.bar(x, vals, color=colors, alpha=0.85, edgecolor='white')
            ax.bar_label(bars, fmt='%.1f%%', fontsize=9)
            ax.set_xticks(x); ax.set_xticklabels(lbls, fontsize=9)
            ax.set_ylim(0, 100); ax.set_title(title); ax.grid(alpha=0.3, axis='y')
        plt.tight_layout(); plt.show()

        # Plot 4: Throughput rolling mean
        fig, ax = plt.subplots(figsize=(12, 3.5))
        for dfi in [df_A, df_B, df_C]:
            lbl  = dfi['strategy'].iloc[0]
            roll = dfi['throughput'].rolling(WIN, min_periods=1).mean()
            ax.plot(dfi['batch'], roll, label=lbl, color=_COL[lbl], lw=1.8)
        ax.set_xlabel('Batch'); ax.set_ylabel('Items / time unit')
        ax.set_title('Throughput (rolling mean)', fontsize=12)
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout(); plt.show()

        # Plot 5: Reorder queue depth (strategy B)
        queue_d, reord_n = [], []
        _, mgr_q = _build_manager(wh_cfg, SW, SW + 100, affinity_store)
        mgr_q.enqueue_all(inventory.cartons, quantity=1)
        _overstock(mgr_q, TF)
        mgr_q.init_lift_state(affinity_store)
        mgr_q.assignment_fn = build_load_minimizing_assignment_fn(
            load_params, affinity_store, wp,
            mgr_q._aisle_sku_sets, mgr_q._aisle_lift_sum, mgr_q._aisle_idx_sets)
        random.seed(SB)
        for i in range(min(NB, 150)):
            triggered = mgr_q.check_reorders()
            queue_d.append(mgr_q.queue_depth)
            reord_n.append(len(triggered))
            batch = Batch(batch_cfg, inventory, affinity=affinity_store)
            tasks = Task.from_batch(batch, mgr_q.warehouse, manager=mgr_q)
            if tasks:
                PickSimulation(tasks, pick_cfg, manager=mgr_q).run()
        fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
        axes[0].plot(queue_d, color='#e05c40', lw=1.5)
        axes[0].set_ylabel('Queued units'); axes[0].grid(alpha=0.3)
        axes[0].set_title('Reorder queue depth — Strategy B (Load-Min)', fontsize=11)
        axes[1].bar(range(len(reord_n)), reord_n, color='#5b9bd5', alpha=0.7)
        axes[1].set_ylabel('SKUs triggered'); axes[1].set_xlabel('Batch')
        axes[1].grid(alpha=0.3, axis='y')
        plt.tight_layout(); plt.show()

    dA_mean = df_A['duration'].mean()
    status.value = (
        f'✅ Done &nbsp;|&nbsp; '
        f'<b>A</b> {dA_mean:.1f} &nbsp;'
        f'<b>B</b> {df_B["duration"].mean():.1f} '
        f'({(df_B["duration"].mean()-dA_mean)/dA_mean*100:+.1f}%) &nbsp;'
        f'<b>C</b> {df_C["duration"].mean():.1f} '
        f'({(df_C["duration"].mean()-dA_mean)/dA_mean*100:+.1f}%)'
    )

btn_run.on_click(run_simulation)
display(out_sim)

In [ ]:
# ── Cell 7 · Sensitivity scan callback ────────────────────────────────────────
def run_sensitivity(_=None):
    SW       = w_seed_world.value
    KP       = w_k_pickers.value
    TF       = w_target_fill.value
    SC       = w_sing_cap.value
    N_SENS   = w_sens_n.value

    lam_vals = np.linspace(w_lam_min.value, w_lam_max.value, w_lam_n.value).round(2)
    gam_vals = np.linspace(w_gam_min.value, w_gam_max.value, w_gam_n.value).round(2)

    pick_cfg  = PickConfig(
        num_pickers      = KP,
        x_move_time      = w_x_move.value,
        y_move_time      = w_y_move.value,
        pick_intercept   = w_intercept.value,
        pick_weight_coef = w_wt_coef.value,
        pick_volume_coef = w_vol_coef.value,
        cart_swap_coef   = w_cart_coef.value,
    )
    wp        = WorkloadParams.from_pick_config(pick_cfg)
    batch_cfg = BatchConfig(n_skus, w_mean_frac.value, w_std_frac.value)
    wh_cfg    = _size_warehouse(SC, TF)

    n_rows, n_cols = len(lam_vals), len(gam_vals)
    grid_b = np.full((n_rows, n_cols), float('nan'))   # B vs A Δ%
    grid_c = np.full((n_rows, n_cols), float('nan'))   # C vs A Δ%
    total  = n_rows * n_cols

    for li, lam in enumerate(lam_vals):
        for gi, gam in enumerate(gam_vals):
            lp = LoadParams(lambda_=float(lam), k=w_k_load.value, gamma=float(gam))

            _, mgr_a = _build_manager(wh_cfg, SW, SW + 100)
            mgr_a.enqueue_all(inventory.cartons, quantity=1); _overstock(mgr_a, TF)

            _, mgr_b = _build_manager(wh_cfg, SW, SW + 100, affinity_store)
            mgr_b.enqueue_all(inventory.cartons, quantity=1); _overstock(mgr_b, TF)
            mgr_b.init_lift_state(affinity_store)
            mgr_b.assignment_fn = build_load_minimizing_assignment_fn(
                lp, affinity_store, wp,
                mgr_b._aisle_sku_sets, mgr_b._aisle_lift_sum, mgr_b._aisle_idx_sets)

            _, mgr_c = _build_manager(wh_cfg, SW, SW + 100, affinity_store)
            mgr_c.enqueue_all(inventory.cartons, quantity=1); _overstock(mgr_c, TF)
            mgr_c.init_lift_state(affinity_store)
            mgr_c.assignment_fn = build_load_maximizing_assignment_fn(
                lp, affinity_store, wp,
                mgr_c._aisle_sku_sets, mgr_c._aisle_lift_sum, mgr_c._aisle_idx_sets)

            random.seed(w_seed_batch.value)
            dA, dB, dC = [], [], []
            for i in range(N_SENS):
                mgr_a.check_reorders(); mgr_b.check_reorders(); mgr_c.check_reorders()
                batch = Batch(batch_cfg, inventory, affinity=affinity_store)
                ta = Task.from_batch(batch, mgr_a.warehouse, manager=mgr_a)
                tb = Task.from_batch(batch, mgr_b.warehouse, manager=mgr_b)
                tc = Task.from_batch(batch, mgr_c.warehouse, manager=mgr_c)
                if not ta or not tb or not tc: continue
                ea = PickSimulation(ta, pick_cfg, manager=mgr_a).run()
                eb = PickSimulation(tb, pick_cfg, manager=mgr_b).run()
                ec = PickSimulation(tc, pick_cfg, manager=mgr_c).run()
                dA.append(extract_batch_stats(ea, i, KP).duration)
                dB.append(extract_batch_stats(eb, i, KP).duration)
                dC.append(extract_batch_stats(ec, i, KP).duration)

            ma, mb, mc = np.mean(dA), np.mean(dB), np.mean(dC)
            if ma > 0:
                grid_b[li, gi] = (mb - ma) / ma * 100
                grid_c[li, gi] = (mc - ma) / ma * 100

            done = li * n_cols + gi + 1
            with out_sens:
                clear_output(wait=True)
                print(f'{done}/{total}  \u03bb={lam:.1f}  \u03b3={gam:.1f}  '
                      f'B\u0394%={grid_b[li,gi]:.2f}  C\u0394%={grid_c[li,gi]:.2f}')

    with out_sens:
        clear_output(wait=True)
        row_labels = [f'\u03bb={v}' for v in lam_vals]
        col_labels = [f'\u03b3={v}' for v in gam_vals]
        vabs = max(5, np.nanmax(np.abs(np.concatenate([grid_b.ravel(), grid_c.ravel()]))))

        fig, axes = plt.subplots(1, 2, figsize=(max(8, n_cols * 2), max(4, n_rows * 1.1)))
        fig.suptitle(f'\u0394% batch duration vs Uniform (A)  [{N_SENS} batches/cell]',
                     fontsize=12, fontweight='bold')
        for ax, grid, title in [
            (axes[0], grid_b, 'B Load-Min'),
            (axes[1], grid_c, 'C Load-Max'),
        ]:
            im = ax.imshow(grid, cmap='RdYlGn_r', vmin=-vabs, vmax=vabs, aspect='auto')
            ax.set_xticks(range(n_cols)); ax.set_xticklabels(col_labels, fontsize=9)
            ax.set_yticks(range(n_rows)); ax.set_yticklabels(row_labels, fontsize=9)
            for i in range(n_rows):
                for j in range(n_cols):
                    ax.text(j, i, f'{grid[i,j]:.1f}%', ha='center', va='center',
                            fontsize=9, color='black')
            plt.colorbar(im, ax=ax, label='\u0394%')
            ax.set_title(f'{title}\n(negative = faster than A)', fontsize=10)
        plt.tight_layout(); plt.show()

        print('\nLoad-Min (B):')
        display(pd.DataFrame(grid_b, index=row_labels, columns=col_labels).round(2))
        print('\nLoad-Max (C):')
        display(pd.DataFrame(grid_c, index=row_labels, columns=col_labels).round(2))

btn_sens.on_click(run_sensitivity)
display(out_sens)